## Imports

In [1]:
import pandas as pd
from pathlib import Path

## Variables globales

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "raw"

In [3]:
dataFrames = {}
for csv_path in Path(DATA_DIR).glob("*.csv"):
    dataFrames[csv_path.stem] = pd.read_csv(csv_path)


## Description:

Dans cette section, nous présentons une description des différentes tables de notre jeu de données brutes.

### Forme:

In [4]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.shape)

-----------links-----------
(260855, 16)
-----------nodes-----------
(95581, 11)
-----------travel_times_2013-----------
(103803579, 5)


Les fichiers travel_times sont de loin les plus lourds, possédant environ 10 millions d'entrées chacuns.

In [5]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.dtypes)


-----------links-----------
link_id              int64
begin_node_id        int64
end_node_id          int64
begin_angle        float64
end_angle          float64
street_length      float64
osm_name            object
osm_class           object
osm_way_id           int64
startX             float64
startY             float64
endX               float64
endY               float64
osm_changeset        int64
birth_timestamp      int64
death_timestamp      int64
dtype: object
-----------nodes-----------
node_id                     int64
is_complete                object
num_in_links                int64
num_out_links               int64
osm_traffic_controller    float64
xcoord                    float64
ycoord                    float64
osm_changeset               int64
birth_timestamp             int64
death_timestamp             int64
grid_region_id              int64
dtype: object
-----------travel_times_2013-----------
begin_node_id      int64
end_node_id        int64
datetime          ob

On observe deux clés utilisées dans notre jeu de données: node_id et link_id. Un champ grid_region_id est également présent mais n'apparait que dans nodes comme clé étrangère.

### échantillons de valeurs:

In [6]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.head())

-----------links-----------
   link_id  begin_node_id  end_node_id  begin_angle  end_angle  street_length  \
0        1      103235840    103225947      -161.51      16.99         84.295   
1        2       42516422     42516427       175.46      -4.68        260.392   
2        3       42516422     42516418        -4.60     175.40        256.804   
3        4      103235530    103235525      -103.43      82.40         76.107   
4        5       42762376     42756156       -23.92     156.25        197.829   

            osm_name    osm_class  osm_way_id     startX     startY  \
0     PalisadeAvenue     tertiary   223694294 -74.021807  40.884127   
1  SchenectadyAvenue  residential     5679907 -73.932264  40.641950   
2  SchenectadyAvenue  residential     5679907 -73.932264  40.641950   
3     HarristownRoad     tertiary    11580026 -74.142573  40.948839   
4    KingslandAvenue  residential     5698286 -73.844794  40.877882   

        endX       endY  osm_changeset  birth_timestamp  d

On peut observer que dans ce dataset, les booléens prennent la forme d'un string prenant les valeurs t/f. 
#### Links
Le champ osm_class semble être une énumération de valeurs. 
#### Nodes 
On confirme également que grid_region_id prends bien la forme d'un identifiant bien qu'il ne soit pour l'instant pas évident de déterminer ce qu'il référence, peut-être simplement un découpage de l'espace ?
Le champ osm_traffic_controller prends des valeurs nulles dans tout l'échantillon.

## Statistiques des valeurs:

### Valeurs nulles:

In [7]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.isnull().sum())

-----------links-----------
link_id               0
begin_node_id         0
end_node_id           0
begin_angle           0
end_angle             0
street_length         0
osm_name           8387
osm_class             0
osm_way_id            0
startX                0
startY                0
endX                  0
endY                  0
osm_changeset         0
birth_timestamp       0
death_timestamp       0
dtype: int64
-----------nodes-----------
node_id                       0
is_complete                   0
num_in_links                  0
num_out_links                 0
osm_traffic_controller    95581
xcoord                        0
ycoord                        0
osm_changeset                 0
birth_timestamp               0
death_timestamp               0
grid_region_id                0
dtype: int64
-----------travel_times_2013-----------
begin_node_id    0
end_node_id      0
datetime         0
travel_time      0
num_trips        0
dtype: int64


Le jeu de données possède des valeurs nulles dans seulement deux champs:
- links.osm_name qui correspondent aux routes sans nom.
- nodes.osm_traffic_controller qui est nul pour toutes les entrées de la table. C'est donc un champ inutile que l'on pourra supprimer.

Au delà de ces deux champs, nos données n'ont aucune valeures manquantes ou incomplètes.

### Valeurs uniques:

In [8]:
for df_name, df in dataFrames.items():
    print("-----------" + df_name + "-----------")
    print(df.nunique())

-----------links-----------
link_id            260855
begin_node_id       96435
end_node_id         96453
begin_angle         35494
end_angle           35517
street_length       99405
osm_name            17825
osm_class              22
osm_way_id          60334
startX              91508
startY              90728
endX                91525
endY                90740
osm_changeset        5813
birth_timestamp         1
death_timestamp         1
dtype: int64
-----------nodes-----------
node_id                   95581
is_complete                   1
num_in_links                 11
num_out_links                12
osm_traffic_controller        0
xcoord                    90708
ycoord                    89952
osm_changeset              4322
birth_timestamp               1
death_timestamp               1
grid_region_id              353
dtype: int64
-----------travel_times_2013-----------
begin_node_id      23535
end_node_id        23891
datetime            8760
travel_time      2005654
num_trips 

#### links:
- osm_class possède 9 valeurs, ce qui comfirme qu'il s'agit bien d'une énumération
Ces valeurs sont:

In [9]:
print(dataFrames["links"]["osm_class"].unique())

['tertiary' 'residential' 'secondary' 'primary' 'motorway' 'motorway_link'
 'trunk' 'trunk_link' 'unclassified' 'secondary_link' 'services'
 'primary_link' 'tertiary_link' 'proposed' 'construction' 'platform'
 'road' 'living_street' 'raceway' 'footway' '1stStreet' 'closed']



- birth et death_timestamp possèdent une seule valeur sur toute la table. Ils ne donnent donc aucune information sur les entrées et sont redondants

#### nodes:
- is_complete ne possède qu'une seule valeur: true, ce qui indique que toutes les entrées de nodes sont complètes, mais rend le champ redondant.
- grid_region_id possède un petit nombre de valeurs, il référence donc soit une petite table, soit un découpage géographique comme supposé plus tôt.

## Relations inter-tables: clé node_id

### Clés orphelines:

In [10]:
primary_node_ids = set(dataFrames["nodes"]["node_id"].unique())

for df_name, df in dataFrames.items():
    if df_name == "nodes":
        continue
    
    print("------------- clés orphelines dans: " + df_name + " -------------")

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id:")
    print(len(df_begin_node_ids - primary_node_ids))

    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id:")
    print(len(df_end_node_ids - primary_node_ids))

------------- clés orphelines dans: links -------------
begin_node_id:
854
end_node_id:
872
------------- clés orphelines dans: travel_times_2013 -------------
begin_node_id:
1
end_node_id:
1


#### links:
La table links contient un nombre non négligeable de clés orphelines, il faudra donc décider de garder ou non ces entrées en fonction de si elles peuvent nuire à la complétude du graphe.

#### travel_times:
Pour les tables travel_times, une seule valeur orpheline est rescencée:

In [11]:
for df_name, df in dataFrames.items():
    if df_name == "nodes" or df_name == "links":
        continue

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id orpheline de " + df_name)
    print(df_begin_node_ids - primary_node_ids)


    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id orpheline de " + df_name)
    print(df_end_node_ids - primary_node_ids)


begin_node_id orpheline de travel_times_2013
{np.int64(0)}
end_node_id orpheline de travel_times_2013
{np.int64(0)}


Toutes ces valeurs orphelines de node_id correspondent donc à l'identifiant 0. C'est un identifiant trop particulier pour que ce soit un simple oubli.
En effet, en regardant de plus près les trajets de travel_times qui partent de la node_id 0, on observe:

In [ ]:
# print(dataFrames["travel_times_2010"][dataFrames["travel_times_2010"]["begin_node_id"] == 0]["end_node_id"].unique())
# print(dataFrames["travel_times_2011"][dataFrames["travel_times_2011"]["begin_node_id"] == 0]["end_node_id"].unique())
# print(dataFrames["travel_times_2012"][dataFrames["travel_times_2012"]["begin_node_id"] == 0]["end_node_id"].unique())
print(dataFrames["travel_times_2013"][dataFrames["travel_times_2013"]["begin_node_id"] == 0]["end_node_id"].unique())



KeyError: 'travel_times_2010'

Et pour ceux qui arrivent à la node_id 0:

In [ ]:
# print(dataFrames["travel_times_2010"][dataFrames["travel_times_2010"]["end_node_id"] == 0]["begin_node_id"].unique())
# print(dataFrames["travel_times_2011"][dataFrames["travel_times_2011"]["end_node_id"] == 0]["begin_node_id"].unique())
# print(dataFrames["travel_times_2012"][dataFrames["travel_times_2012"]["end_node_id"] == 0]["begin_node_id"].unique())
print(dataFrames["travel_times_2013"][dataFrames["travel_times_2013"]["end_node_id"] == 0]["begin_node_id"].unique())


Cette valeur de node_id "orpheline" semble donc en réalité être une valeur par défaut attribuée aux trajets qui se passent sur des routes indéterminées, puisque tous les trajets contenant cette clé orpheline, se déplacent entre la node 0 et 0.

### Clées manquantes:

In [ ]:
for df_name, df in dataFrames.items():
    if df_name == "nodes":
        continue
    
    print("------------- clés manquantes dans: " + df_name + " -------------")

    df_begin_node_ids = set(df["begin_node_id"].unique())
    print("begin_node_id:")
    print(len(primary_node_ids - df_begin_node_ids))

    df_end_node_ids = set(df["end_node_id"].unique())
    print("end_node_id:")
    print(len(primary_node_ids - df_end_node_ids))

#### Dans links:

Dans la table links, aucune clé manquante n'est recensée, tous les sommets de graphe sont donc reliés.

#### Dans travel_times:
Les clés manquantes elles, sont moins problèmatiques, puisqu'elles révèlent juste la non complétude de certaines informations.
Il faut tout de même souligner que ces valeurs manquantes dans travel_times indique que nous n'avons pas de données sur le traffic de bon nombre de routes de la ville.